# Gizli Dirichlet Dağılımı (Latent Dirichlet Allocation) (LDA)

🎯 Bu challenge'ın amacı, **LDA** algoritması (NLP'de Denetimsiz Öğrenme) ile e-posta külliyatı içinde konular bulmaktır.

✉️ İşte 1000'den fazla ***etiketlenmemiş e-posta*** içeren bir koleksiyon. Bunlardan ***konuları çıkarmaya*** çalışalım!

In [1]:
import pandas as pd

url = 'https://d32aokrjazspmn.cloudfront.net/materials/lda_data'

data = pd.read_csv(url, sep=",", header=None)
data.columns = ['text']
data.head()

,text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...


In [2]:
data.shape

(1199, 1)

## (1) Preprocessing 

❓ **Question (Cleaning**) ❓ You're used to it by now... Clean up! Store the cleaned text in a new column "clean_text" of the DataFrame.

In [3]:
import re
import string
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocessing(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))

    words = word_tokenize(text)

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words and len(word) > 2
    ]

    return " ".join(words)

data['clean_text'] = data['text'].apply(preprocessing)

data[['text', 'clean_text']].head()

[nltk_data] Downloading package punkt to
[nltk_data]     /home/burak_gokce/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/burak_gokce/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/burak_gokce/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/burak_gokce/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,text,clean_text
0,From: gld@cunixb.cc.columbia.edu (Gary L Dare)...,gldcunixbcccolumbiaedu gary dare subject stan ...
1,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlepvelaacsoaklandedu cardinal ximenez sub...
2,From: miner@kuhub.cc.ukans.edu\nSubject: Re: A...,minerkuhubccukansedu subject ancient book orga...
3,From: atterlep@vela.acs.oakland.edu (Cardinal ...,atterlepvelaacsoaklandedu cardinal ximenez sub...
4,From: vzhivov@superior.carleton.ca (Vladimir Z...,vzhivovsuperiorcarletonca vladimir zhivov subj...


## (2) Latent Dirichlet Allocation model

❓ **Soru (Eğitim)** ❓ Potansiyel konuları çıkarmak için bir LDA modeli eğitin

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

vectorizer = CountVectorizer(
    max_df=0.95,
    min_df=2,
    max_features=5000
)

X_bow = vectorizer.fit_transform(data['clean_text'])

lda_model = LatentDirichletAllocation(
    n_components=5,
    random_state=42,
    max_iter=10
)

lda_model.fit(X_bow)

print("BoW matris boyutu:", X_bow.shape)
print("Keşfedilen konu sayısı:", lda_model.n_components)

BoW matris boyutu: (1199, 5000)
Keşfedilen konu sayısı: 5


##  (3) Potansiyel konuları görselleştirin

🎁 Potansiyel konularla ilişkili kelimeleri yazdırmak için bir  fonksiyon kodladık.

In [5]:
def print_topics(model, vectorizer):
    for idx, topic in enumerate(model.components_):
        print("Topic %d:" % (idx))
        print([(vectorizer.get_feature_names_out()[i], topic[i])
                        for i in topic.argsort()[:-10 - 1:-1]])

❓ **Soru** ❓ LDA tarafından çıkarılan konuları yazdırın.

In [6]:
print_topics(lda_model, vectorizer)

Topic 0:
[('god', 1463.3848907124332), ('christian', 746.8974265786841), ('one', 724.7409662697089), ('would', 681.3384754717468), ('people', 627.2854703870561), ('jesus', 622.8619464467757), ('say', 478.3340341692044), ('know', 437.7344613882597), ('think', 433.39411185731575), ('organization', 430.07278642747576)]
Topic 1:
[('church', 347.78734171404176), ('catholic', 154.89596514715458), ('organization', 131.2276413634348), ('one', 117.4925850131808), ('pope', 113.7356999133551), ('would', 104.99810274263812), ('marriage', 99.05300464343905), ('european', 97.41916538289645), ('like', 94.88243121356447), ('get', 90.16885633602207)]
Topic 2:
[('game', 471.4393831425274), ('team', 376.15863368055005), ('player', 328.0014973154849), ('would', 299.0039731692158), ('organization', 288.0383141455342), ('writes', 258.3734093720434), ('think', 241.61607630433707), ('one', 235.9511647093828), ('university', 210.68358379382727), ('article', 190.59339615215555)]
Topic 3:
[('team', 470.389728788

## (4) Yeni bir metnin belge-konu karışımını tahmin edin

❓ **Soru (Tahmin)** ❓

LDA modeliniz fit edildiğine göre, onu yeni bir metnin konularını tahmin etmek için kullanabilirsiniz.

1. Örneği vektörleştirin
2. Vektörleştirilmiş örnek üzerinde LDA'yı kullanarak konuları tahmin edin

In [7]:
example = ["My team performed poorly last season. Their best player was out injured and only played one game"]

In [8]:
clean_example = [preprocessing(text) for text in example]

example_bow = vectorizer.transform(clean_example)

topic_distribution = lda_model.transform(example_bow)

for topic_id, probability in enumerate(topic_distribution[0]):
    print(f"Topic {topic_id}: {probability:.2%}")

print("En baskın konu:", topic_distribution[0].argmax())

Topic 0: 1.84%
Topic 1: 1.84%
Topic 2: 51.47%
Topic 3: 1.85%
Topic 4: 43.01%
En baskın konu: 2


🏁 Tebrikler! LDA'yı hızlı bir şekilde nasıl uygulayacağınızı öğrendiniz.

💾 Not defterinizi `git add/commit/push` yapmayı unutmayın...

🚀 ... ve bir sonraki göreve geçin!